In [1]:
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

import sys

sys.path.append("../..")

In [2]:
from src.data.load_data import load_data
from src.models.kernel_regression import KernelRegression
from src.conformal_prediction.oracle_cp import OracleConformalPredictor

Load data

In [3]:
input_points, output_points = load_data("friedman1")

In [4]:
train_input_points, test_input_points, train_output_points, test_output_points = (
    train_test_split(input_points, output_points, random_state=0)
)

Instantiate predictor

In [5]:
# loss_name = "log_cosh"
# loss_params = {"alpha":1.}

loss_name = "pseudo_huber"
loss_params = {"alpha": 1.0}

# loss_name = "smoothed_pinball"
# loss_params = {"alpha":1., "tau":0.5}

In [6]:
predictor = KernelRegression(
    solver="lbfgs", loss_name=loss_name, loss_params=loss_params, lam=0.5
)

Instantiate region predictor

In [7]:
conformal_predictor = OracleConformalPredictor(predictor, non_conformity_name="absolute")
region_predictor = conformal_predictor.fit_predict(
    train_input_points, train_output_points,
    test_input_points, test_output_points
)

In [8]:
confidence_control_level = 0.1
prediction_regions = region_predictor(confidence_control_level)

In [ ]:
coverage = np.mean(
    [
        test_output_point in prediction_region
        for test_output_point, prediction_region in zip(
            test_output_points, prediction_regions
        )
    ]
)
print("test coverage: ", coverage)

test coverage:  0.896
